# Linear regression MSE loss

In [1]:
# Std Libs
from enum import StrEnum

# Third Party Libs
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn

# Local
from src.data.loader import get_enb_data
from src.optim_ml.optimizers.norm_constrained_gd import (
    QdfCauchyFrobeniusGD,
    QdfCauchySpectralGD,
)

# Data

## Load Dataset

In [2]:
features_subset = 1
df_X, df_y = get_enb_data(subset=features_subset)

# Scale
df_X = (df_X - df_X.mean(axis=0))/df_X.std(axis=0)

# Tensors
X = torch.from_numpy(df_X.values).float() # input
Y = torch.from_numpy(df_y.values).float() # actual output

# shapes
n1, d = df_X.shape
n2, p = df_y.shape
n = n1

Read: C:\Users\aishema\Documents\CodeProjects\Education\ml-projects\src\data\datasets\ENB2012_data.csv
Features: (768, 8)
Targets: (768, 2)


## Describes

In [3]:
df_X.describe().T

,count,mean,std,min,25%,50%,75%,max
X1,768.0,0.000000e+00,1.0,-1.362924,-0.772061,-0.133929,0.622376,2.040447
X3,768.0,0.000000e+00,1.0,-1.684757,-0.561586,0.000000,0.561586,2.246342
X6,768.0,0.000000e+00,1.0,-1.340767,-0.670384,0.000000,0.670384,1.340767
X7,768.0,7.401487e-17,1.0,-1.759300,-1.008666,0.117287,1.243239,1.243239
X8,768.0,-7.401487e-17,1.0,-1.813393,-0.685060,0.120893,0.765655,1.410417


In [4]:
df_corrs = df_X.corr().round(4)
df_corrs.abs().style.background_gradient(cmap ='viridis', axis=None)

,X1,X3,X6,X7,X8
X1,1.000000,0.203800,0.000000,0.000000,0.000000
X3,0.203800,1.000000,0.000000,0.000000,0.000000
X6,0.000000,0.000000,1.000000,0.000000,0.000000
X7,0.000000,0.000000,0.000000,1.000000,0.213000
X8,0.000000,0.000000,0.000000,0.213000,1.000000


In [5]:
G = X.T @ X
H = (2/(n*p)) * G

rank_X = np.linalg.matrix_rank(X)
rank_G = np.linalg.matrix_rank(G)
cond_nb_X = np.linalg.cond(X)
cond_nb_G = np.linalg.cond(G)

is_X_full_column_rank = d == rank_X

spec_h = np.linalg.eigh(H).eigenvalues

lambda_min, lambda_max = np.min(spec_h), np.max(spec_h)
mu = lambda_min
L = lambda_max
kappa = L/mu

dict_ranks = {
    "mode": features_subset,
    "features": "|".join(df_X.columns.tolist()),
    "count_features": d,
    "rank_X": rank_X,
    "is_X_full_column_rank": is_X_full_column_rank,
    "rank_G": rank_G,
    "is_G_full_rank": d == rank_G,
    "cond_nb_G": cond_nb_G,
    "L": L,
    "mu": mu,
    "kappa": kappa,
}

df_mode = pd.DataFrame([dict_ranks])
df_mode.T

,0
mode,1
features,X1|X3|X6|X7|X8
count_features,5
rank_X,5
is_X_full_column_rank,True
rank_G,5
is_G_full_rank,True
cond_nb_G,1.541181
L,1.211384
mu,0.78601


## Multiple Output Linear Regression

### Objective function


<u> Docs Pytorch MSE Loss </u>

$$\ell(x, y) = L = \{l_1,\dots,l_N\}^\top, \quad
 |      l_n = \left( x_n - y_n \right)^2$$


$$\ell(x, y) =
  \begin{cases}
 \operatorname{mean}(L) &  \text{if reduction} = \text{`mean'}\\
 \operatorname{sum}(L)  &  \text{if reduction} = \text{`sum'}
 \end{cases}$$

<u> Definition </u>

Let
- $n, d, p \in \mathbb{N}^*$
- $X \in \mathcal{M}_{n,d}(\mathbb{R})$ features
- $Y \in \mathcal{M}_{n,p}$ targets
- $\forall \beta = (\beta_{i, j})_{(i,j) \in ⟦1,d⟧ \times ⟦1,p⟧} \in \mathcal{M}_{d,p}(\mathbb{R}),\; \|\beta\|_F^2 = \displaystyle \sum_{i,j} \beta_{i,j}^2$ the Frobenius norm

The pytorch MSE loss objective function is such that $\forall \beta \in \mathcal{M}_{d,p}(\mathbb{R})$

$$F(\beta) = \frac{1}{np}\|X\beta - Y\|^2$$

The Frobenius inner product on matrices:

$$\langle A,B\rangle_F=\operatorname{tr}(A^\top B)=\sum_{i,j}A_{ij}B_{ij}=\operatorname{vec}(A)^\top\operatorname{vec}(B)$$

Since the objective is $F(\beta)=\frac{1}{np}\|X\beta-Y\|_F^2$ (the norm $\|\cdot\|$ in the notebook is the Frobenius 
norm defined at 01_well_cond.ipynb:339), this is exactly the Euclidean/dot product once matrices are flattened to vectors.

In [6]:
# Here we check the formula of the value returned by the torch.nn.MSELoss
mat_1 = torch.ones(Y.shape)
mat_2 = Y

objective_func = torch.nn.MSELoss(
    reduction='mean',
)
value_1 = objective_func(mat_1, mat_2)
value_2 = (1/(n*p)) * ((mat_1 - mat_2) ** 2).sum(axis=0).sum() 
value_2 = np.round(value_2, 3)

delta = abs(value_1 - value_2)
delta

tensor(0.)

### Objective function properties

$F$ is twice differentiable and

$$\nabla F(\beta) = \frac{2}{np}X^T(X\beta-Y) \quad \text{and} \quad \nabla^2 F(\beta) = \frac{2}{np}X^TX := H$$

Since $\forall \beta \in \mathcal{M}_{d,p}(\mathbb{R}), \nabla^2 F(\beta) \succeq 0$, F is convex therefore a minimizer $\beta^*$ of $F$ is such that

$$\nabla F(\beta^*) = \frac{2}{np}X^T(X\beta^*-Y) = 0$$

i.e.

$$X^TX\beta^* = X^TY$$


<u>Exact solution</u>


$X$ has full column rank $\iff$ $X^\top X$ has full rank

In this case $X^\top X$ is invertible and $\beta^*$ is such that $\beta^* = (X^\top X)^{-1}X^\top Y$

Therefore $X\beta^* = X(X^\top X)^{-1}X^\top Y := PY$ With $P = X(X^\top X)^{-1}X^\top; \quad P^2 = P; \quad P^\top = P$

The orthogonal projection matrix on $\text{span}$ of the columns of $X$


<u>Approximation of solution by gradient descent</u>


When $X$ does not have full rank, $X$ has full rank but a bad condition number, $X^\top X$ is too large to invert easily ($d \gg 1$, *complexity $\mathcal{O}(d^3)$ or using Cholesky $\mathcal{O}(\frac{d^3}{3})$*). 

In some cases regularized methods (Lasso, Ridge, Elastic Net) give the minimizer of a penalized objective, which can be preferable to OLS. But in all these cases an approximation of a minimizer $\beta^*$ obtained by GD can still be very satisfying and requires no inversion of $X^\top X$. 

However, when the conditioning is bad, convergence can be quite slow. 

One can notice that:

- $\forall \beta \in \mathcal{M}_{d,p}(\mathbb{R}),\; \nabla^2F(\beta) \succeq 0$
- $F$ is $L$-smooth with $L = \frac{2\lambda_{\max}(X^\top X)}{np}$
- If $\lambda_{\min}(X^\top X) > 0$, $F$ is $\mu$-strongly convex with
  $\mu = \frac{2\lambda_{\min}(X^\top X)}{np}$
- $\mu>0$ so
   $\kappa=\frac{L}{\mu} =\frac{\lambda_{\max}(X^\top X)}{\lambda_{\min}(X^\top X)}$

### OLS Solution

In [7]:
assert is_X_full_column_rank 
beta_star_ols = np.linalg.inv(G) @ X.T @ Y
Y_pred_ols = X @ beta_star_ols

loss_fn = torch.nn.MSELoss()
mse_ols = float(loss_fn(Y, Y_pred_ols).detach().numpy())
pd.DataFrame([mse_ols], columns=['mse_ols'])

,mse_ols
0,570.946472


## Gradient Descent Methods

### Optimizer kwargs

In [8]:
class GdSequence(StrEnum):
    classic_gd = "classicGD"
    hb = "HB"
    nag = "NAG"
    cauchy_frobenius_gd = "cauchyFrobeniusGD"
    cauchy_spectral_gd = "cauchySpectralGD"

In [9]:
epsilon = 1e-5
epsilon

1e-05

In [10]:
# GD
lr_gd = 2/(mu + L)
epochs_gd = np.log((2 * epsilon)/(L))/(2 * np.log((kappa - 1)/(kappa + 1)))
epochs_gd = int(np.ceil(epochs_gd))

# HB
lr_hb = 4/((np.sqrt(L) + np.sqrt(mu)) ** 2)
mom_hb = ((np.sqrt(kappa) - 1)/(np.sqrt(kappa) + 1)) ** 2
delta = 1e-5
epochs_hb = np.log(epsilon) / (
    2 * np.log((np.sqrt(kappa) - 1)/(np.sqrt(kappa) + 1) + delta)
)
epochs_hb = int(np.ceil(epochs_hb))

# NAG
lr_nag = 1/L
mom_nag = (np.sqrt(kappa) - 1)/(np.sqrt(kappa) + 1)
epochs_nag = np.log(2*epsilon/(L+mu))/np.log(1 - 1/np.sqrt(kappa))
epochs_nag = int(np.ceil(epochs_nag))

# Cauchy Frobenius GD
epochs_cfgd = np.log((2 * epsilon)/(L))/(2 * np.log((kappa - 1)/(kappa + 1)))
epochs_cfgd = int(np.ceil(epochs_cfgd)) 

# Cauchy Spectral GD
epochs_csgd = epochs_cfgd

# QdfCauchyFrobeniusGD

list_disp_op = [
    {
        "features_subset": features_subset,
        "method": GdSequence.classic_gd,
        "kappa": kappa,
        "lr": lr_gd, 
        "mom": None, 
        "delta": None, 
        "asymptotic_rate": (kappa - 1)/(kappa+1),
        "epsilon": epsilon, 
        "epochs": epochs_gd,
    },
    {
        "features_subset": features_subset,
        "method": GdSequence.hb,
        "kappa": kappa,  
        "lr": lr_hb, 
        "mom": mom_hb, 
        "delta": None, 
        "asymptotic_rate": (np.sqrt(kappa) - 1)/(np.sqrt(kappa) + 1),
        "epsilon": epsilon, 
        "epochs": epochs_hb,
    },
    {
        "features_subset": features_subset,
        "method": GdSequence.nag,
        "kappa": kappa,  
        "lr": lr_nag, 
        "mom": mom_nag, 
        "asymptotic_rate": 1 - 1/np.sqrt(kappa),
        "delta": delta, 
        "epsilon": epsilon, 
        "epochs": epochs_nag,
    },
    {
            "features_subset": features_subset,
            "method": GdSequence.cauchy_frobenius_gd, 
            "kappa": kappa,
            "lr": "Not fixed", 
            "mom": None, 
            "delta": None, 
            "asymptotic_rate": (kappa - 1)/(kappa+1), 
            "epsilon": epsilon, 
            "epochs": epochs_cfgd,
    },
    {
        "features_subset": features_subset,
        "method": GdSequence.cauchy_spectral_gd, 
        "kappa": kappa,
        "lr": "Not fixed", 
        "mom": None, 
        "delta": None, 
        "asymptotic_rate": (kappa - 1)/(kappa+1), 
        "epsilon": epsilon, 
        "epochs": epochs_csgd,
    },
]

df_disp_op = pd.DataFrame(list_disp_op)
df_disp_op

,features_subset,method,kappa,lr,mom,delta,asymptotic_rate,epsilon,epochs
0,1,classicGD,1.541181,1.001305,NaN,NaN,0.212964,0.00001,4
1,1,HB,1.541181,1.012923,0.011603,NaN,0.107718,0.00001,3
2,1,NAG,1.541181,0.825502,0.107718,0.00001,0.194486,0.00001,8
3,1,cauchyFrobeniusGD,1.541181,Not fixed,NaN,NaN,0.212964,0.00001,4
4,1,cauchySpectralGD,1.541181,Not fixed,NaN,NaN,0.212964,0.00001,4


In [11]:
# # Epochs: We use the max epoch to observe the convergence of all methods
# epochs = int(np.ceil(np.max([
#     epochs_gd, epochs_hb, epochs_nag, epochs_cfgd, epochs_csgd
# ])))
# print(f"Epochs: {epochs}")

In [12]:

dict_kwargs_opt = {
    GdSequence.classic_gd: {
        'opt_name': 'SGD',
        'epochs': epochs_gd,
        'opt_kwargs': {'lr': lr_gd},
    },
    GdSequence.hb: {
        'opt_name': 'SGD',
        'epochs': epochs_hb,
        'opt_kwargs': {'lr': lr_hb, 'momentum': mom_hb},
    },
    GdSequence.nag: {
        'opt_name': 'SGD',
        'epochs': epochs_nag,
        'opt_kwargs': {
            'lr': lr_nag,
            'momentum': mom_nag,
            'nesterov': True,
        },
    },
    GdSequence.cauchy_frobenius_gd: {
        'opt_name': 'QdfCauchyFrobeniusGD',
        'epochs': epochs_cfgd,
        'opt_kwargs': {'hessian': H},
    },
    GdSequence.cauchy_spectral_gd: {
        'opt_name': 'QdfCauchySpectralGD',
        'epochs': epochs_csgd,
        'opt_kwargs': {'hessian': H},
    },
}
dict_kwargs_opt.keys()

dict_keys([<GdSequence.classic_gd: 'classicGD'>, <GdSequence.hb: 'HB'>, <GdSequence.nag: 'NAG'>, <GdSequence.cauchy_frobenius_gd: 'cauchyFrobeniusGD'>, <GdSequence.cauchy_spectral_gd: 'cauchySpectralGD'>])

### Run GD

In [13]:
def run_gd(
    **kwargs,
):
    # Tensors
    X_ts = X
    Y_ts = Y
    
    # Create Model
    n_samples, in_features = X.shape
    out_features = 1 if len(Y.shape) == 1 else Y.shape[1]
    print(f'In features: {in_features}')
    print(f'Out features: {out_features}')

    model = nn.Linear(
        in_features=in_features, 
        out_features=out_features, 
        bias=False, 
    )

    # Initial weights
    w0 = np.ones((X_ts.shape[1], Y_ts.shape[1]), dtype='float32')
    w0 = torch.as_tensor(w0, dtype=torch.float32).reshape(model.weight.shape)
    with torch.no_grad():
        model.weight.copy_(w0)

    # Loss / Objective function
    criterion = torch.nn.MSELoss()
    epochs = kwargs['epochs']

    # 
    kwargs_optimizer = {
        **{'params': model.parameters()},
        **kwargs['opt_kwargs'], 
    }
    opt_name = kwargs['opt_name']
    if opt_name == 'SGD':
         optimizer = torch.optim.SGD(**kwargs_optimizer)
    elif opt_name == 'MUON':
         optimizer = torch.optim.Muon(**kwargs_optimizer)
    elif opt_name == 'QdfCauchySpectralGD':
        optimizer = QdfCauchySpectralGD(**kwargs_optimizer)
    elif opt_name == 'QdfCauchyFrobeniusGD':
        optimizer = QdfCauchyFrobeniusGD(**kwargs_optimizer)
    else:
        raise ValueError(f'Unexpected opt_name: {opt_name}')
    
    list_weights = []
    list_loss = []

    # Train model
    for epoch in range(epochs):

        # forward pass
        Y_pred = model(X_ts)
        loss = criterion(Y_pred, Y_ts)
        list_loss.append({'epoch': epoch, 'loss': loss.item()})

        # backward pass
        loss.backward()
        optimizer.step()

        # empty the gradient
        optimizer.zero_grad()

        # model weights
        weights = model.weight.detach().numpy().ravel().copy()
        list_weights.append(weights)

        # display advancement
        if ((epoch + 1) % 10 == 0) or ((epoch + 1) in [1, epochs]):
            print(f"epoch: {epoch+1}, loss: {loss.item():4f}")
            

    Y_pred_gd = model(X_ts).detach().numpy()[:, 0]

    weights = np.array(list_weights)

    dict_res = {
        'X': X_ts,
        'Y': Y_ts,
        'model': model,
        'weights': weights,
        'y_pred_gd': Y_pred_gd,
        'loss': list_loss,

    }
    return dict_res


dict_res_opts = {}
for name in dict_kwargs_opt:
    print(f'{name} Optimizer')
    dict_res_opts[name] = run_gd(**dict_kwargs_opt[name])
    print('+' * 50)

classicGD Optimizer
In features: 5
Out features: 2
epoch: 1, loss: 625.020569
epoch: 4, loss: 570.950317
++++++++++++++++++++++++++++++++++++++++++++++++++
HB Optimizer
In features: 5
Out features: 2
epoch: 1, loss: 625.020569
epoch: 3, loss: 570.990784
++++++++++++++++++++++++++++++++++++++++++++++++++
NAG Optimizer
In features: 5
Out features: 2
epoch: 1, loss: 625.020569
epoch: 8, loss: 570.946472
++++++++++++++++++++++++++++++++++++++++++++++++++
cauchyFrobeniusGD Optimizer
In features: 5
Out features: 2
epoch: 1, loss: 625.020569
epoch: 4, loss: 570.946594
++++++++++++++++++++++++++++++++++++++++++++++++++
cauchySpectralGD Optimizer
In features: 5
Out features: 2
epoch: 1, loss: 625.020569
epoch: 4, loss: 571.004272
++++++++++++++++++++++++++++++++++++++++++++++++++


## Loss

### Loss function at each steps

In [14]:
df_loss = None
list_traces = []
fig = go.Figure()
for name in dict_res_opts:
    list_loss = dict_res_opts[name]['loss']
    df_loss_temp = pd.DataFrame(list_loss)
    fig.add_trace(go.Scatter(
        x=df_loss_temp['epoch'], y=df_loss_temp['loss'], name=f'loss_{name}',
    ))

# OLS loss value
fig.add_hline(
    y=float(mse_ols),
    line_width=3,
    line_dash="dash",
    line_color="black",
    annotation_text="loss_OLS",
)
# fig.add_hline(
#     y=float(mse_ols),
#     line_width=3,
#     line_dash="dash",
#     line_color="black",
#     annotation_text="OLS",
# )

# Title, Axes Titles
fig.update_layout(
    title=(
        f'Gradient Descent Methods Loss - Features Subset '
        f'{features_subset} - kappa={kappa:.2f}'
    ),
    xaxis=dict(title=dict(text="epochs",)),
    yaxis=dict(title=dict(text="loss",)),  
    width=900, 
    height=450,
    legend=dict(
        orientation="h",
        x=0.5,
        y=-0.2,
        xanchor="center",
        yanchor="top"
    )
)
fig.show()

### Loss function last values

In [15]:
dict_loss = {
    **{"features_subset": features_subset},
    **{'OLS': mse_ols},
    **{
        name: dict_res_opts[name]['loss'][-1]['loss']
        for name in dict_res_opts 
    }
}
df_disp_loss = pd.DataFrame([dict_loss])
df_disp_loss

,features_subset,OLS,classicGD,HB,NAG,cauchyFrobeniusGD,cauchySpectralGD
0,1,570.946472,570.950317,570.990784,570.946472,570.946594,571.004272


### Distance between solutions

In [16]:
dict_beta_star = {
    name: dict_res_opts[name]['model'].weight.detach().numpy().copy().T 
    for name in dict_res_opts 
}
display(dict_beta_star)
dict_mse = {
    name: ((beta_star_ols.numpy() - dict_beta_star[name]) ** 2).mean(axis=0).mean()
    for name in dict_res_opts
}
dict_mse = {
    **{"features_subset": features_subset},
    **dict_mse,
}
df_disp_beta_star = pd.DataFrame([dict_mse])
df_disp_beta_star

,features_subset,classicGD,HB,NAG,cauchyFrobeniusGD,cauchySpectralGD
0,1,0.00004,0.000179,4.237375e-10,8.948751e-07,0.000552
